# Multi-agent : sélection d'actifs et optimisation de portefeuille

**Le LLM analyse, le code calcule, le gérant décide.**

Ce notebook suit le schéma d'architecture de l'équipe : on déroule d'abord les **9 étapes une par une**
(pour voir ce que produit chaque nœud), puis on lance le **graphe LangGraph complet**, qui s'arrête sur la
**validation du gérant** (`interrupt()`) et reprend avec sa décision (`Command(resume=...)`).

| Étape | Type | Nœud LangGraph | Fichier |
|---|---|---|---|
| 1 Construire l'univers | Code | `charger_mandat`, `construire_univers` | `src/nodes/preparation.py` |
| 2 Collecter les données | Code | `collecter_donnees` (RetryPolicy) | `src/nodes/preparation.py` |
| 3 Calculer les métriques | Code | `calculer_metriques` → `Send` × N | `src/nodes/preparation.py` |
| 4 Analyser chaque titre | LLM × 3 en parallèle | sous-graphe `analyse_titre` | `src/agents/analystes.py` |
| 5 Débattre et sélectionner | LLM | sous-graphe `debat` + `juge_selection` | `src/agents/debat.py` |
| 6 Optimiser les pondérations | Code | `optimiser` (Ledoit-Wolf, Black-Litterman) | `src/nodes/portefeuille.py` |
| 7 Contrôler les risques | Code + LLM | `controle_risque` (3 essais max, H9) | `src/nodes/portefeuille.py` |
| 8 Valider | **Humain** | `validation_gerant` (`interrupt`) | `src/nodes/validation.py` |
| 9 Rapport et journal | Code + LLM | `rapport` (Store = journal d'audit) | `src/nodes/rapport.py` |

Tout est **gratuit et open source** : LangGraph / LangChain, LLM Qwen2.5 via **Ollama**, PyPortfolioOpt,
yfinance (prototype), DuckDuckGo pour la recherche web, FinBERT non requis.
Sans GPU : `LLM_BACKEND = "mock"` (règles déterministes, quelques secondes) — le graphe est identique.

> Projet pédagogique : ce n'est pas un conseil en investissement.

## 0. Installation (Google Colab)

1. *Exécution → Modifier le type d'exécution → GPU T4* (pour le LLM).
2. Récupérer le projet : `REPO_URL` (GitHub) ou, s'il est vide, import de `portfolio-agents.zip`.

In [ ]:
import os, sys, shutil, subprocess, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""   # ex. "https://github.com/<equipe>/portfolio-agents.git"

def bonne_version(root: Path) -> bool:
    """Vrai si le dossier contient la version du code qui correspond à CE notebook."""
    f = root / "src" / "state.py"
    return f.exists() and "EtatPortefeuille" in f.read_text(encoding="utf-8")

if IN_COLAB:
    os.chdir("/content")
    ROOT = Path("/content/portfolio-agents")
    if ROOT.exists() and not bonne_version(ROOT):
        print("Ancienne version du projet détectée : suppression.")
        shutil.rmtree(ROOT)
    if not ROOT.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
        else:
            from google.colab import files
            zip_name = next(iter(files.upload()))          # choisir portfolio-agents.zip
            subprocess.run(["unzip", "-q", "-o", zip_name, "-d", "/content"], check=True)
            os.remove(zip_name)
    if not bonne_version(ROOT):
        raise RuntimeError("Ce zip ne correspond pas à ce notebook : utilise le notebook qui est DANS le zip.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Oublier tout code « src » déjà chargé (évite de mélanger ancienne et nouvelle version sans redémarrer)
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Racine du projet :", ROOT, "| version OK :", bonne_version(ROOT))

### LLM open source avec Ollama

Ollama sert Qwen2.5 localement **avec appels d'outils et sorties structurées**, ce dont `create_agent` a besoin.
Sur un T4 : `qwen2.5:3b` (3 analystes, nombreux appels) et `qwen2.5:7b` (débat, juge, rapport).

In [ ]:
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False

LLM_BACKEND = "ollama" if (IN_COLAB and HAS_GPU) else "mock"   # forcer "mock" pour une démo instantanée
DATA_SOURCE = "yfinance"                                        # ou "synthetic" (hors-ligne)
MAX_TITRES = 15 if LLM_BACKEND != "mock" else 50                # titres envoyés aux analystes (temps LLM)

if LLM_BACKEND == "ollama" and IN_COLAB:
    if subprocess.run("which ollama", shell=True, capture_output=True).returncode != 0:
        # zstd : requis par l'installateur Ollama récent ; pciutils/lshw : détection du GPU
        subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw", shell=True, check=True)
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    for m in ["qwen2.5:3b", "qwen2.5:7b"]:
        subprocess.run(["ollama", "pull", m], check=True)
print("LLM :", LLM_BACKEND, "| GPU :", HAS_GPU, "| titres analysés :", MAX_TITRES)

## 1. Observabilité LangSmith (optionnel, gratuit)

LangSmith enregistre **chaque nœud, chaque agent, chaque appel d'outil et de modèle** : durée, entrées, sorties,
erreurs. Utile pour comprendre pourquoi un agent a donné telle note, ou où le cycle a ralenti.

1. Crée un compte gratuit sur [smith.langchain.com](https://smith.langchain.com) → *Settings → API Keys* → crée une clé.
2. Dans Colab, clique sur l'icône 🔑 (**Secrets**) à gauche → ajoute `LANGSMITH_API_KEY` avec ta clé → active
   « Accès au notebook ».
3. Exécute la cellule ci-dessous. Sans clé, tout fonctionne pareil, sans traces.

In [ ]:
try:
    from google.colab import userdata
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    print("Clé LangSmith trouvée dans les Secrets Colab")
except Exception:
    print("Pas de clé LangSmith : le notebook fonctionne sans traces")
os.environ.setdefault("LANGSMITH_PROJECT", "portfolio-agents")

## 2. Configuration

- `config/settings.yaml` : réglages **techniques** (modèles, poids des 3 notes, κ, δ, stratégie des 3 essais…)
- `config/mandat.yaml` : le **mandat** du client (H5 : 10 %/titre, 30 %/secteur, liquidité, limites de risque)

In [ ]:
import pandas as pd
from IPython.display import HTML, Markdown, display
from src.config import AppContext, load_settings

pd.set_option("display.precision", 3); pd.set_option("display.max_colwidth", 90)
settings = load_settings(overrides={"llm": {"backend": LLM_BACKEND},
                                    "donnees": {"fournisseur": DATA_SOURCE, "repli_synthetique": True},
                                    "analyse": {"max_titres": MAX_TITRES}})
ctx = AppContext.build(settings)
DATE = None   # None = dernière séance ; ex. "2026-10-31" pour rejouer une date passée (point-in-time)

o = settings["optimisation"]
print("Poids des notes :", settings["analyse"]["poids_notes"])
print(f"Black-Litterman : kappa = {o['kappa']} (rendement par point de vue), delta = {o['delta']} (aversion au risque du marché)")
print("Essais d'optimisation (H9) :", [t["objectif"] for t in o["tentatives"]])
from src.observabilite import langsmith_actif, url_projet
print("LangSmith :", f"actif, projet {os.environ['LANGSMITH_PROJECT']} -> {url_projet()}" if langsmith_actif() else "inactif")

## 3. Organisation des fichiers

In [ ]:
SKIP = {"__pycache__", ".git", ".ipynb_checkpoints", "cache", ".pytest_cache"}
def tree(path, prefix=""):
    items = sorted([p for p in path.iterdir() if p.name not in SKIP], key=lambda p: (p.is_file(), p.name))
    for i, p in enumerate(items):
        last = i == len(items) - 1
        print(prefix + ("└── " if last else "├── ") + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir() and p.name != "fixtures":
            tree(p, prefix + ("    " if last else "│   "))
print(ROOT.name + "/"); tree(ROOT)

## 4. Le contrat partagé : `EtatPortefeuille`

Chaque nœud lit l'état et renvoie **uniquement ses clés**. `notes` et `journal` ont un *reducer*
(`operator.add`) : les 3 × N notes produites en parallèle sont **fusionnées**, pas écrasées.

In [ ]:
import typing
from src.state import EtatPortefeuille, NoteAnalyste, Selection
for k, v in typing.get_type_hints(EtatPortefeuille, include_extras=True).items():
    print(f"{k:18s} {getattr(v, '__name__', v)}")
print("\nNoteAnalyste :", list(NoteAnalyste.model_fields))
print("Selection    :", list(Selection.model_fields), "->", list(Selection.model_fields["titres"].annotation.__args__[0].model_fields))

---
# Partie A — les 9 étapes, une par une

On appelle directement les fonctions des nœuds, dans l'ordre du graphe. `etat` joue le rôle de l'état LangGraph.

In [ ]:
from src.local_runner import _apply      # fusion des mises à jour (reducers notes / journal)
from src.nodes import preparation, portefeuille, validation, rapport
etat = {"date_analyse": DATE, "notes": [], "journal": []}

def etape(fn, *args):
    upd = fn(etat, ctx, *args)
    _apply(etat, upd)
    for line in upd.get("journal", []):
        print(line)

## Étape 1 — Construire l'univers *(code · H2, H5)*
`lire_mandat()` → `get_constituants()` → `filtrer_univers()` : liquidité, capitalisation, exclusions, **avec motif**.

In [ ]:
etape(preparation.charger_mandat)
etape(preparation.construire_univers)
display(pd.Series(etat["mandat"]["contraintes"]).to_frame("contrainte du mandat"))
pd.DataFrame(etat["univers"]["exclus"]) if etat["univers"]["exclus"] else print("Aucune exclusion pour ce mandat")

## Étape 2 — Collecter les données *(code · H6, H8)*
Prix ajustés (3 ans) **coupés à la date d'analyse**, fondamentaux, actualités. La version des données est
une empreinte des prix : elle est écrite dans le journal d'audit (H10).

In [ ]:
etape(preparation.collecter_donnees)
d = etat["donnees"]
print("Version des données :", d["version"]); print("Sources :", d["sources"])
t0 = etat["univers"]["eligibles"][0]
pd.DataFrame(d["actualites"][t0])[["published", "title", "source"]].head(5)

## Étape 3 — Calculer les métriques *(code · H7)*
Rendements, volatilité, momentum, bêta, PER, ROE, endettement… et **scores standardisés** (z-scores bornés à ±3 ;
valorisation comparée **à l'intérieur du secteur**). Aucun LLM ici.

In [ ]:
etape(preparation.calculer_metriques)
met = pd.DataFrame(etat["metriques"]).T
cols = ["secteur", "rendement_1a", "volatilite", "sharpe", "beta", "per", "roe", "z_qualite", "z_valorisation", "z_momentum", "score_quant"]
met[cols].sort_values("score_quant", ascending=False).head(10)

## Étape 4 — Analyser chaque titre *(LLM · 3 agents en parallèle · H7)*

Dans le graphe, `calculer_metriques` émet un **`Send` par titre** vers le sous-graphe `analyse_titre`, où
`agent_fondamental`, `agent_sentiment` et `agent_valorisation` tournent **en parallèle**. Chacun est un
`create_agent` LangChain avec ses outils (`get_metriques_titre`, `rechercher_rapport`, `get_actualites`,
`recherche_web`, `comparer_pairs`) et renvoie une `NoteAnalyste` (sortie structurée Pydantic).

Voici le contenu d'un `Send` et les 3 notes d'un titre :

In [ ]:
from src.agents.analystes import analyse_titre_sans_graphe
payloads = preparation.payloads_analyse(etat, ctx)
p = payloads[0]
print(f"{len(payloads)} Send ; exemple pour {p['ticker']} : clés {list(p)}")
pd.DataFrame(analyse_titre_sans_graphe(p, ctx)["notes"])[["agent", "score", "these", "risques", "sources"]]

In [ ]:
# Ici en séquentiel pour la lisibilité ; dans le graphe, en parallèle (max_concurrency).
t = time.time()
for i, p in enumerate(payloads, 1):
    _apply(etat, analyse_titre_sans_graphe(p, ctx))
    if i % 5 == 0: print(f"{i}/{len(payloads)} titres analysés ({time.time() - t:.0f} s)")
notes = pd.DataFrame(etat["notes"])
notes.pivot_table(index="ticker", columns="agent", values="score").assign(
    moyenne=lambda d: d.mean(axis=1)).sort_values("moyenne", ascending=False).head(12)

## Étape 5 — Débattre et sélectionner *(LLM · H10)*

Sous-graphe `debat` : le **chercheur haussier** défend les titres prometteurs, le **chercheur baissier** cherche
les failles (2 tours), puis le **juge** renvoie une `Selection` : liste restreinte (15 à 30 titres), **vue** (−2..+2)
et **confiance** par titre. Le code applique ensuite des garde-fous (tickers valides, taille, diversification).

In [ ]:
etape(lambda s, c: __import__("src.agents.debat", fromlist=["x"]).executer_debat(s, c, avec_graphe=False))
for m in etat["debat"]:
    print(f"--- {m['role'].upper()} (tour {m['tour']}) ---\n{m['texte'][:600]}\n")
pd.DataFrame(etat["vues"]).T

## Étapes 6 et 7 — Optimiser puis contrôler les risques *(code + LLM · H4, H5, H9)*

**Optimiser** : covariance **Ledoit-Wolf** → a priori de marché `π = δ·Σ·w_marché` → vues du juge
`Q = π + κ·vue` avec confiance (Idzorek) → **Black-Litterman** → **frontière efficiente** sous contraintes du mandat.

**Contrôler** : conformité, VaR/CVaR, concentration, rotation, stress tests ; le LLM rédige l'explication.
Non conforme → nouvel essai plus prudent (3 au maximum), puis escalade au gérant **avec alerte**.

In [ ]:
etat["iterations_risque"] = 0
route = "optimiser"
while route == "optimiser":
    etape(portefeuille.optimiser)
    etape(portefeuille.controle_risque)
    route = portefeuille.router_risque(etat, ctx)
print("\n->", route)
bl = pd.DataFrame(etat["attendus"]["rendements"]).T
bl["poids"] = pd.Series(etat["poids"])
bl.sort_values("poids", ascending=False)

In [ ]:
rr = etat["rapport_risque"]
print(rr["explication"], "\n")
display(pd.Series(rr["mesures"]).to_frame("mesure"))
pd.Series(rr["stress"]).to_frame("perte estimée")

## Étape 8 — Valider *(humain · H1)*

Dans le graphe, `validation_gerant` appelle `interrupt(demande)` : l'exécution **s'arrête** et l'état est
sauvegardé par le checkpointer. Voici ce que voit le gérant, puis sa réponse.

In [ ]:
demande = validation.demande_validation(etat)
print({k: demande[k] for k in ("conforme", "alerte", "problemes", "attendus")})
DECISION = {"decision": "approuve", "motif": "conforme au comité"}
# DECISION = {"decision": "modifie", "poids": {...}, "motif": "..."}   -> retour au contrôle des risques
# DECISION = {"decision": "rejete", "motif": "..."}                     -> fin du cycle, motif journalisé
_apply(etat, validation.appliquer_decision(etat, ctx, DECISION))
print(etat["journal"][-1])

## Étape 9 — Rapport et journal *(code + LLM · H1, H10)*
`calculer_mouvements()` (ordres à passer **manuellement**), puis la **note d'investissement** : un fichier HTML
autonome mis en page pour l'A4 (tableaux et graphiques calculés par le code, texte rédigé par le LLM), et le
journal d'audit dans le **Store** : thèse, sources, version des données, version du mandat.

In [ ]:
import html as _html
if etat["decision_gerant"]["decision"] == "approuve":
    etape(rapport.rapport)
    doc = Path(etat["rapport_fichier"]).read_text(encoding="utf-8")
    display(HTML(f'<iframe srcdoc="{_html.escape(doc)}" style="width:100%;height:1100px;border:0"></iframe>'))
    print("Fichier :", etat["rapport_fichier"], "— ouvre-le puis « Imprimer → Enregistrer en PDF »")

In [ ]:
if IN_COLAB and etat.get("rapport_fichier"):
    from google.colab import files
    files.download(etat["rapport_fichier"])      # télécharge la note sur ton ordinateur

In [ ]:
from src.tools.reporting import lire_journal
journal = lire_journal(ctx.store, etat["date_analyse"])
pd.DataFrame([{"evenement": e["evenement"], "horodatage": e["horodatage"], "cles": ", ".join(k for k in e if k not in ("evenement", "horodatage"))} for e in journal])

---
# Partie B — le graphe LangGraph complet

Même logique, assemblée dans `src/graph.py` : `Send`, sous-graphes, `RetryPolicy`, boucle de risque,
`interrupt()` / `Command(resume=...)`, checkpointer et Store.

In [ ]:
from src.graph import build_graph, config_cycle, etat_cycle, lancer_cycle, reprendre, interruption_en_cours
ctx_g = AppContext.build(settings)          # Store neuf pour ce cycle
app = build_graph(ctx_g)
try:
    from IPython.display import Image
    display(Image(app.get_graph(xray=1).draw_mermaid_png()))      # rendu via mermaid.ink (internet)
except Exception:
    print(app.get_graph(xray=1).draw_mermaid())                    # à coller dans https://mermaid.live

In [ ]:
cfg = config_cycle(ctx_g, thread_id=f"cycle-demo-{int(time.time())}")
t = time.time()
demande = lancer_cycle(app, cfg, DATE)          # s'arrête sur interrupt() (étape 8)
print(f"\n⏸ En attente du gérant ({time.time() - t:.0f} s) — prochain nœud : {app.get_state({'configurable': cfg['configurable']}).next}")
print({k: demande[k] for k in ("conforme", "alerte", "problemes", "attendus")})
pd.DataFrame({"poids": demande["poids"]}).join(pd.DataFrame(demande["vues"]).T[["vue", "confiance"]])

Le gérant répond. Essayez aussi `modifie` (avec des poids) : le graphe repasse par `controle_risque`,
puis revient vous demander validation.

In [ ]:
DECISION = {"decision": "approuve", "motif": "validé en comité"}
suite = reprendre(app, cfg, DECISION)            # Command(resume=DECISION)
if suite:
    print("Nouvelle demande de validation :", suite["problemes"])
else:
    final = etat_cycle(app, cfg)
    print("Note :", final["rapport_fichier"])

from src.graph import liens_traces
for l in liens_traces(cfg):                       # liens directs vers chaque trace LangSmith du cycle
    print(f"Trace {l['etape']:22s}", l["url"] or f"(LangSmith inactif, run {l['run_id'][:8]})")

In [ ]:
# Checkpointer : tout l'historique du cycle est conservé (reprise après panne, « time travel »)
hist = list(app.get_state_history({"configurable": cfg["configurable"]}))
print(len(hist), "checkpoints ; derniers nœuds exécutés :", [h.next for h in hist[:6]])

## Visualisation

La courbe est calculée **dans l'échantillon** (mêmes données que l'analyse) : c'est une illustration,
pas un backtest. Un backtest point-in-time (H8) relancerait le graphe à chaque fin de mois avec `DATE`.

In [ ]:
import matplotlib.pyplot as plt
from src.nodes.preparation import prix_frame
from src.tools.metrics import portfolio_returns

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#1f1f1e", "#6b6a63"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": "#e6e5e0",
                     "grid.linewidth": 0.6, "font.size": 10})
w = pd.Series(etat["poids"]).sort_values()
fig, ax = plt.subplots(figsize=(7, 0.3 * len(w) + 1))
ax.barh(w.index, w.values, color=BLUE, height=0.6)
for y, v in enumerate(w.values):
    ax.text(v + 0.002, y, f"{v:.1%}", va="center", color=INK, fontsize=8)
ax.axvline(etat["mandat"]["contraintes"]["poids_max_titre"], color=MUTED, lw=1, ls="--")
ax.set_title("Allocation cible (pointillés : plafond par titre du mandat)", loc="left", color=INK)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}")); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

In [ ]:
px = prix_frame(etat["donnees"]); r = px.pct_change().dropna()
port = (1 + portfolio_returns(etat["poids"], r)).cumprod()
bench = (1 + r[etat["donnees"]["benchmark"]]).cumprod()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(port.index, port, color=BLUE, lw=2, label="Portefeuille proposé")
ax.plot(bench.index, bench, color=ORANGE, lw=2, label=f"Indice ({etat['donnees']['benchmark']})")
for s, c in ((port, "Portefeuille"), (bench, "Indice")):
    ax.annotate(c, (s.index[-1], s.iloc[-1]), xytext=(4, 0), textcoords="offset points", va="center", color=INK, fontsize=9)
ax.legend(frameon=False, loc="upper left"); ax.grid(axis="x", visible=False)
ax.set_title("Valeur de 1 investi — dans l'échantillon", loc="left", color=INK)
plt.tight_layout(); plt.show()

## Tests (hors-ligne : données synthétiques + LLM mock)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-2000:] or r.stderr[-2000:])

## 🔗 Lancer le site du comité avec un lien public

La cellule lance l'application Streamlit en arrière-plan puis ouvre un tunnel **Cloudflare** (gratuit, sans compte).
Ouvre le lien affiché, choisis le moteur des agents dans le panneau de gauche, puis **Lancer l'analyse**.
Le site reprend le moteur et la clé LangSmith de ce notebook. Le lien fonctionne tant que la session Colab reste ouverte.

In [ ]:
import re
if not Path("/usr/local/bin/cloudflared").exists():
    subprocess.run("wget -q -O /usr/local/bin/cloudflared "
                   "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
                   "&& chmod +x /usr/local/bin/cloudflared", shell=True, check=True)
subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)   # relance propre
os.environ["LLM_BACKEND"] = LLM_BACKEND                                           # moteur proposé par défaut
subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app/streamlit_app.py", "--server.port", "8501",
                  "--server.headless", "true"], cwd=ROOT,
                 stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                 stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0); break
print("🔗 Lien public :", url or "pas encore prêt : relance la cellule ou lis /content/tunnel.log")
print("En cas d'erreur sur la page :", "print(open('/content/streamlit.log').read())")

## Points encore ouverts (schéma, « choix techniques encore ouverts »)

- **Fournisseur de données** : yfinance ne convient qu'au prototype ; EODHD / FMP ou Bourse de Casablanca + AMMC pour le Maroc.
- **Modèle LLM et budget** : qwen2.5 3b/7b sur T4 ; mesurer le temps par titre avant de passer à 300 titres.
- **Base vectorielle du RAG** : TF-IDF par défaut ; sentence-transformers, puis Chroma/FAISS si le corpus grossit.
- **Modifié → contrôle des risques** : si les poids du gérant sont non conformes, le schéma renvoie à `optimiser`
  (qui remplace ses poids) — à confirmer avec les gérants.